In [0]:
%run ./import_libraries

In [0]:
%run ./schema_registry

In [0]:
# Read control_load table to get list of tables to process
control_df = spark.table("adwm_wh.utilities.control_load")

# Filter for active tables only
active_tables = control_df.filter(col("isactive") == "Y").collect()

# print(f"Found {len(active_tables)} active tables to process:")
# for row in active_tables:
#     print(f"  - {row['schema']}.{row['table']}")

In [0]:
# Configuration
VOLUME_BASE_PATH = "/Volumes/adwm_wh/volumes/landing_files"
CHECKPOINT_BASE_PATH = "/Volumes/adwm_wh/volumes/checkpoints/bronze"  # Checkpoint location for streaming state
CATALOG = "adwm_wh"
BRONZE_SCHEMA = "bronze"

# Process each active table with Auto Loader streaming
processed_tables = []
failed_tables = []
streaming_queries = []

for row in active_tables:
    database = row['database']
    schema_name = row['schema']
    table_name = row['table']
    table_key = f"{schema_name}.{table_name}".lower()
    
    try:
        # Construct paths
        volume_path = f"{VOLUME_BASE_PATH}/{database}/{schema_name}/*/{table_name}/"
        checkpoint_path = f"{CHECKPOINT_BASE_PATH}/{schema_name}_{table_name}"
        bronze_table = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
        
        # print(f"\nProcessing: {table_key}")
        # print(f"  Source: {volume_path}")
        # print(f"  Target: {bronze_table}")
       
        # Read data with Auto Loader (cloudFiles) - STREAMING mode
        raw_df = (
            spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("cloudFiles.schemaLocation", checkpoint_path)  # Store inferred schema
            .option("header", "true")
            .schema(schema_registry[table_key])  # Enforce expected schema
            .load(volume_path)
            .selectExpr(
                "*",
                "_metadata.file_name as source_file_name",
                "_metadata.file_path as source_file_path",
                "_metadata.file_modification_time as source_file_timestamp",
                "current_timestamp() as ingestion_timestamp"
            )
        )
        
        # Write to bronze Delta table with streaming
        query = (
            raw_df.writeStream
            .format("delta")
            .outputMode("append")
            .option("checkpointLocation", checkpoint_path)
            .option("mergeSchema", "true")  # Allow schema evolution if needed
            .trigger(availableNow=True)  # Process all available files then stop
            .toTable(bronze_table)
        )
        
        # Wait for this table's streaming query to complete
        query.awaitTermination()
        
        print(f"  ✓ Successfully processed {table_key}")
        processed_tables.append(table_key)
        streaming_queries.append((table_key, query.id))
        
    except Exception as e:
        print(f"  ✗ Error processing {table_key}: {str(e)}")
        failed_tables.append((table_key, str(e)))

print(f"\n{'='*60}")
print(f"Processing Summary:")
print(f"  Successfully processed: {len(processed_tables)} tables")
print(f"  Failed: {len(failed_tables)} tables")

if failed_tables:
    print(f"\nFailed tables:")
    for table, error in failed_tables:
        print(f"  - {table}: {error[:100]}...")  # Truncate long errors

if processed_tables:
    print(f"\nSuccessfully processed tables written to {CATALOG}.{BRONZE_SCHEMA}:")
    for table in processed_tables:
        print(f"  - {table}")

In [0]:
%sql
-- Verify data was loaded into bronze tables
-- Check one table as example
-- SELECT * FROM adwm_wh.bronze.Address
-- LIMIT 10;